In [ ]:
#This code blockprovides the full LLM prompt builder




prompt_first_half = """
You are an expert in the PostGIS spatial database and highly skilled in writing SQL queries for geographic analysis.

PostGIS extends PostgreSQL with spatial data types and functions.
Below is a quick reference of commonly used spatial functions and their meanings:

- ST_Within(geomA, geomB): Returns TRUE if geometry A is completely within geometry B.
- ST_Intersects(geomA, geomB): Returns TRUE if geometry A and B share any portion of space.
- ST_Distance(geomA, geomB): Returns the minimum distance between geometries A and B, in the same units as the spatial reference system.
- ST_Length(geometry): Returns the length of a LineString in its spatial reference units.
- ST_Area(geometry): Returns the area of a Polygon.
- ST_Contains(geomA, geomB): Returns TRUE if geometry A contains geometry B.
- ST_Union(geomA, geomB): Returns a geometry that represents the union of geomA and geomB.
- ST_Centroid(geometry): Returns the geometric center of a geometry.
- ST_Buffer(geometry, distance): Returns a geometry covering all points within a given distance from the input geometry.
- ST_AsText(geometry): Returns the Well-Known Text (WKT) representation of the geometry.

Below are example pairs of natural-language questions (NLQs) and their corresponding SQL queries:

"""

prompt_second_half = """

Below is the database schema:

CREATE TABLE counties (
    "geometry" geometry(GEOMETRY,4326),
    "statefp" BIGINT,
    "countyfp" BIGINT,
    "countyns" BIGINT,
    "geoid" BIGINT,
    "name" TEXT,
    "namelsad" TEXT,
    "lsad" BIGINT,
    "classfp" TEXT,
    "mtfcc" TEXT,
    "csafp" TEXT,
    "cbsafp" TEXT,
    "metdivfp" TEXT,
    "funcstat" TEXT,
    "aland" BIGINT,
    "awater" BIGINT,
    "intptlat" DOUBLE PRECISION,
    "intptlon" DOUBLE PRECISION,
    "state" TEXT,
    "stateaabrv" TEXT
);

Decription of table columns:

geometry - polygon for each county
statefp - state FIPS code
countyfp - county FIPS code
countyns - county GNIS/ANSI code
geoid - combined state+county FIPS
name - county name
namelsad - full county name including legal/statistical descriptor
lsad - legal/statistical area descriptor code
classfp - county class code
mtfcc - MAF/TIGER feature class code
csafp - central statistical area FIPS code
cbsafp - core-based statistical area FIPS code
metdivfp - metropolitan division FIPS code
funcstat - functional status of county (active/inactive)
aland - land area in square meters
awater - water area in square meters
intptlat - internal point latitude (centroid)
intptlon - internal point longitude (centroid)
state - full state name
stateaabrv - state two-letter abbreviation


Answer the following with no explanation:

NLQ: """


# -------- Retrieval for a new query --------
def retrieve_topk(query, schema_tokens, embeddings, k=3):
    masked_query, _ = mask_schema_tokens(query, schema_tokens)
    query_emb = model.encode(masked_query, convert_to_tensor=True, normalize_embeddings=True)

    # Cosine similarity
    cos_scores = torch.nn.functional.cosine_similarity(query_emb, torch.stack(df['masked_embedding'].tolist()))
    topk_scores, topk_indices = torch.topk(cos_scores, k=k)

    results = []
    for score, idx in zip(topk_scores, topk_indices):
        idx = idx.item()  # <-- convert tensor to int
        results.append({
            "Query": df.iloc[idx]['Query'],
            "SQL": df.iloc[idx]['Answer'],
            "Score": score.item()
        })
    return results

def builder(user_query):
  user_query = user_query.lower()
  user_query = re.sub(r'[^\w\s]', '', user_query)
  top_examples = retrieve_topk(user_query, schema_tokens, embeddings, k=3)
  full_prompt = ""
  for i, ex in enumerate(top_examples, 1):
      full_prompt += f"[Example-{i}]\n"
      #lowercase ex['Query']
      ex['Query'] = ex['Query'].lower()
      #remove puntuation from ex['Query']
      ex['Query'] = re.sub(r'[^\w\s]', '', ex['Query'])
      full_prompt += "NLQ: " + ex['Query'] + "\n"
      full_prompt += "SQL: " + ex['SQL'] + "\n"
      print("NLQ: " + ex['Query'] + "\n")
      print("SQL: " + ex['SQL'] + "\n")
      print("SCORE: " + str(ex['Score']) + "\n")




  return prompt_first_half + full_prompt + prompt_second_half + user_query + "\n"


In [ ]:
#This code block contains the embedder used in both the full training data and the user query
#This code block can be combinded with the next cell to have it all in one cell on embedding the data and

!pip install -U sentence-transformers transformers --no-cache-dir
import pandas as pd
from sentence_transformers import SentenceTransformer
import re
import torch
import spacy
df = pd.read_csv("/content/Training_Data.csv")


# Load NLP model for POS tagging
nlp = spacy.load("en_core_web_sm")

questions = df['Query'].tolist()


schema_tokens = [
    "geometry", "statefp", "countyfp", "countyns", "geoid", "name", "namelsad",
    "lsad", "classfp", "mtfcc", "csafp", "cbsafp", "metdivfp", "funcstat",
    "aland", "awater", "intptlat", "intptlon", "state", "stateaabrv"
]

state_tokens = [
    "Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado",
    "Connecticut", "Delaware", "Florida", "Georgia", "Hawaii", "Idaho",
    "Illinois", "Indiana", "Iowa", "Kansas", "Kentucky", "Louisiana",
    "Maine", "Maryland", "Massachusetts", "Michigan", "Minnesota",
    "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada",
    "New Hampshire", "New Jersey", "New Mexico", "New York",
    "North Carolina", "North Dakota", "Ohio", "Oklahoma", "Oregon",
    "Pennsylvania", "Rhode Island", "South Carolina", "South Dakota",
    "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington",
    "West Virginia", "Wisconsin", "Wyoming"
]
print(len(state_tokens))
model = SentenceTransformer("all-MiniLM-L6-v2")


def mask_schema_tokens(question, schema_tokens, use_pos_weight=True):
    doc = nlp(question)
    masked_tokens = []
    token_weights = []

    for token in doc:
        tok_text = token.text
        # Mask schema token if it matches column name
        if any(re.fullmatch(t, tok_text, flags=re.IGNORECASE) for t in schema_tokens) or any(re.fullmatch(t, tok_text, flags=re.IGNORECASE) for t in state_tokens):
            masked_tokens.append("[MASK]")
        else:
            masked_tokens.append(tok_text)

        # POS weighting
        if use_pos_weight:
            if token.pos_ in ["NOUN", "PROPN", "NUM"]:
                token_weights.append(1.0)
            else:
                token_weights.append(0.0)
        else:
            token_weights.append(1.0)

    masked_question = " ".join(masked_tokens)
    return masked_question, token_weights




50


In [ ]:
# Mask all questions
masked_questions = []
token_weights_list = []

for q in questions:
    masked_q, weights = mask_schema_tokens(q, schema_tokens)
    masked_questions.append(masked_q)
    token_weights_list.append(weights)

# -------- Embeddings --------
embeddings = model.encode(masked_questions, convert_to_tensor=True, normalize_embeddings=True)
df['masked_embedding'] = list(embeddings)
#write new df to training_data.csv
#df.head()
#df.to_csv("training_data.csv", index=False)

In [ ]:
!pip install flask pyngrok

# Imports
from flask import Flask, request, jsonify
from pyngrok import ngrok
import threading
import time


# Flask app
app = Flask(__name__)
port = 5130  # Use port >1024 for Windows compatibility

# ngrok auth
#token removed for security
auth_token = ''
ngrok.set_auth_token(auth_token)

# Open HTTP tunnel (bind_tls=False avoids Windows SSL issues)
public_url = ngrok.connect(port, bind_tls=False).public_url
print(f"\n * ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:{port}\"\n")
app.config["BASE_URL"] = public_url

# Flask routes
@app.route('/')
def hello():
    return "Testing 1,2,3, testing."

@app.route('/prompt', methods=['POST'])
def handle_prompt():
    data = request.get_json()
    prompt = data.get("prompt", "")
    print("Prompt sent in:", prompt)
    returner = builder(prompt)
    print(returner)
    return jsonify({"received": returner})

# Run Flask in background thread
threading.Thread(target=lambda: app.run(port=port, use_reloader=False)).start()

while(1):
  time.sleep(60)


 * ngrok tunnel "http://nasopharyngeal-thriftily-stevie.ngrok-free.dev" -> "http://127.0.0.1:5130"

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5130
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [11/Dec/2025 03:38:21] "POST /prompt HTTP/1.1" 200 -


Prompt sent in: What is the distance between Vermont and Wisconsin
NLQ: what the distance between tennessee and utah

SQL: SELECT MIN(ST_DISTANCE(ST_TRANSFORM(a.geometry, 5070), ST_TRANSFORM(b.geometry, 5070))) / 1000 as distance_km FROM counties AS a JOIN counties AS b ON b.state = 'Tennessee' WHERE a.state = 'Utah';

SCORE: 0.9793452024459839

NLQ: what the distance between california and wyoming

SQL: SELECT MIN(ST_DISTANCE(ST_TRANSFORM(a.geometry, 5070), ST_TRANSFORM(b.geometry, 5070))) / 1000 as distance_km FROM counties AS a JOIN counties AS b ON b.state = 'Wyoming' WHERE a.state = 'California';

SCORE: 0.9793452024459839

NLQ: what the distance between the states ohio and florida

SQL: SELECT MIN(ST_DISTANCE(ST_TRANSFORM(a.geometry, 5070), ST_TRANSFORM(b.geometry, 5070))) / 1000 as distance_km FROM counties AS a JOIN counties AS b ON b.state = 'Ohio' WHERE a.state = 'Florida';

SCORE: 0.9283131957054138


You are an expert in the PostGIS spatial database and highly skilled in wr

INFO:werkzeug:127.0.0.1 - - [11/Dec/2025 03:39:35] "POST /prompt HTTP/1.1" 200 -


Prompt sent in: Show all border counties of Georgia
NLQ: show all border counties of florida

SQL: SELECT a.* FROM counties AS a JOIN counties AS b ON ST_TOUCHES(a.geometry, b.geometry) WHERE a.state = 'Florida' AND b.state != 'Florida';

SCORE: 1.0

NLQ: show all border counties in california

SQL: SELECT a.* FROM counties a JOIN counties b ON ST_TOUCHES(a.geometry, b.geometry) AND a.state = 'California' AND b.state != 'California';

SCORE: 0.9881049990653992

NLQ: show counties that border colorado

SQL: SELECT a.* FROM counties AS a JOIN counties AS b ON ST_TOUCHES(a.geometry, b.geometry) WHERE a.state != 'Colorado' AND b.state = 'Colorado';

SCORE: 0.9573474526405334


You are an expert in the PostGIS spatial database and highly skilled in writing SQL queries for geographic analysis.

PostGIS extends PostgreSQL with spatial data types and functions.
Below is a quick reference of commonly used spatial functions and their meanings:

- ST_Within(geomA, geomB): Returns TRUE if geometry

INFO:werkzeug:127.0.0.1 - - [11/Dec/2025 03:41:33] "POST /prompt HTTP/1.1" 200 -


... (8 outputs left out)

Prompt sent in: What counties have three words in their name
NLQ: which counties in california have twoword names

SQL: SELECT * FROM counties WHERE state = 'California' AND name LIKE '% %';

SCORE: 0.8951130509376526

NLQ: which counties have multiword names in washington

SQL: SELECT * FROM counties WHERE state = 'Washington' AND name LIKE '% %';

SCORE: 0.8915402293205261

NLQ: which counties have multiword names in minnesota

SQL: SELECT * FROM counties WHERE state = 'Minnesota' AND name LIKE '% %';

SCORE: 0.8915402293205261


You are an expert in the PostGIS spatial database and highly skilled in writing SQL queries for geographic analysis.

PostGIS extends PostgreSQL with spatial data types and functions.
Below is a quick reference of commonly used spatial functions and their meanings:

- ST_Within(geomA, geomB): Returns TRUE if geometry A is completely within geometry B.
- ST_Intersects(geomA, geomB): Returns TRUE if geometry A and B share any portion of space.
- ST_Distance(ge

INFO:werkzeug:127.0.0.1 - - [11/Dec/2025 03:50:38] "POST /prompt HTTP/1.1" 200 -


Prompt sent in: Show border counties of Kansas
NLQ: show all border counties of florida

SQL: SELECT a.* FROM counties AS a JOIN counties AS b ON ST_TOUCHES(a.geometry, b.geometry) WHERE a.state = 'Florida' AND b.state != 'Florida';

SCORE: 0.9767279028892517

NLQ: show counties that border colorado

SQL: SELECT a.* FROM counties AS a JOIN counties AS b ON ST_TOUCHES(a.geometry, b.geometry) WHERE a.state != 'Colorado' AND b.state = 'Colorado';

SCORE: 0.973455548286438

NLQ: show all border counties in california

SQL: SELECT a.* FROM counties a JOIN counties b ON ST_TOUCHES(a.geometry, b.geometry) AND a.state = 'California' AND b.state != 'California';

SCORE: 0.9584946036338806


You are an expert in the PostGIS spatial database and highly skilled in writing SQL queries for geographic analysis.

PostGIS extends PostgreSQL with spatial data types and functions.
Below is a quick reference of commonly used spatial functions and their meanings:

- ST_Within(geomA, geomB): Returns TRUE if

KeyboardInterrupt: 